# Dynamic Multi-Agent Clinical MCQ Evaluation

One fresh API experiment on 57 development and 300 locked test cases. It compares the single GPT-4.1 baseline with JSD, Kendall W, Krippendorff Alpha, and CARE. Dynamic routing assigns GPT-5-mini reasoning only to important or complex roles. GPT-5 reviews final test outcomes for safety. Completed runs are never reused; an interrupted run resumes its own saved work until its final table is written.


In [3]:
# Cell 0 - Agent/model configuration and protected data
from pathlib import Path
from datetime import datetime, timezone
import importlib, json, sys
from itertools import combinations
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def find_root(start=Path.cwd()):
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "evaluation" / "final_agreement_experiment.py").exists():
            return candidate
    raise FileNotFoundError("Project root not found")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import evaluation.final_agreement_experiment as experiment_module
importlib.reload(experiment_module)
from evaluation.final_agreement_experiment import (
    COMPLEX_REASONING_MODEL, FINAL_SAFETY_MODEL, FINAL_SAFETY_REASONING_EFFORT,
    MODEL, ROUTER_MODEL, TEMPERATURE, FinalAgreementExperiment, dataset_identity,
    load_fixed_cases, load_project_api_key,
)

MAX_CONCURRENCY = 16
RUN_ROOT = ROOT / "agreement-experiment" / "fresh_runs"
ACTIVE_RUN_FILE = ROOT / "agreement-experiment" / "ACTIVE_FRESH_RUN.txt"
RESUME_CURRENT_RUN = False
if ACTIVE_RUN_FILE.exists():
    candidate = Path(ACTIVE_RUN_FILE.read_text(encoding="utf-8").strip())
    if candidate.exists() and not (candidate / "final_main_comparison.csv").exists():
        OUTPUT_DIR = candidate
        RUN_ID = candidate.name
        RESUME_CURRENT_RUN = True
if not RESUME_CURRENT_RUN:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    OUTPUT_DIR = RUN_ROOT / RUN_ID
    ACTIVE_RUN_FILE.parent.mkdir(parents=True, exist_ok=True)
    ACTIVE_RUN_FILE.write_text(str(OUTPUT_DIR), encoding="utf-8")
METHODS = ("jsd", "kendall_w", "krippendorff_alpha", "care_consensus")

agent_models = pd.DataFrame([
    ["Independent single", MODEL, "0.0", "Every case", "Baseline final answer"],
    ["Clinical router", ROUTER_MODEL, "low reasoning", "Every case", "Difficulty, importance, lead specialty"],
    ["Routine specialists", MODEL, "0.0", "Simple/moderate routine", "Independent domain opinions"],
    ["Lead reasoning specialist", COMPLEX_REASONING_MODEL, "medium/high", "Complex/high/critical", "Decisive complex reasoning"],
    ["Dynamic adjudicator", f"{MODEL} / {COMPLEX_REASONING_MODEL}", "0.0 / medium/high", "Triggered disagreement", "Final multi-agent answer"],
    ["Final safety reviewer", FINAL_SAFETY_MODEL, FINAL_SAFETY_REASONING_EFFORT, "300 test final answers only", "Safety metric only"],
], columns=["Agent", "Model", "Sampling/Reasoning", "When used", "Task"])
display(agent_models)

CASES = load_fixed_cases(ROOT)
assert len(CASES[CASES.split == "development"]) == 57
assert len(CASES[CASES.split == "test"]) == 300
assert CASES.fixed_case_id.is_unique
print("Dataset identity:", dataset_identity(CASES))
print("Development / test:", 57, "/", 300)
print("Output:", OUTPUT_DIR)
print("Resuming interrupted run:", RESUME_CURRENT_RUN)


,Agent,Model,Sampling/Reasoning,When used,Task
0,Independent single,gpt-4.1,0.0,Every case,Baseline final answer
1,Clinical router,gpt-5-mini,low reasoning,Every case,"Difficulty, importance, lead specialty"
2,Routine specialists,gpt-4.1,0.0,Simple/moderate routine,Independent domain opinions
3,Lead reasoning specialist,gpt-5-mini,medium/high,Complex/high/critical,Decisive complex reasoning
4,Dynamic adjudicator,gpt-4.1 / gpt-5-mini,0.0 / medium/high,Triggered disagreement,Final multi-agent answer
5,Final safety reviewer,gpt-5,medium,300 test final answers only,Safety metric only


Dataset identity: 04e330e0590d6c5ea7c8e64d4f63ccecce5e79b5ca30ade932383f9269939ddb
Development / test: 57 / 300
Output: C:\Users\Taghreed Al-Sharafi\Desktop\New folder\V1_20-80-MCQ-Evaluation-Package-One-GPT5-Judge-FINAL\Multi-Agent-AI-for-Reliable-Clinical-Reasoning\agreement-experiment\fresh_runs\20260819T005742_616204Z
Resuming interrupted run: False


In [4]:
# Cell 1 - Always run fresh API calls; cross-run cached outputs are disabled
key = load_project_api_key(ROOT)
assert key and len(key) > 20
print("API key loaded from .env (hidden). Fresh run:", RUN_ID)
if "experiment" not in globals() or getattr(experiment, "output_dir", None) != OUTPUT_DIR.resolve():
    experiment = FinalAgreementExperiment(
        project_root=ROOT,
        output_dir=OUTPUT_DIR,
        max_concurrency=MAX_CONCURRENCY,
        reuse_existing_outputs=RESUME_CURRENT_RUN,
    )
else:
    print("Resuming this fresh run in the current kernel after an interruption.")
shared = await experiment.ensure_shared_outputs()
assert len(shared) == 357
await experiment.evaluate_single()
for method in METHODS:
    print("Evaluating:", method)
    await experiment.evaluate_method(method)

required = ["single_predictions.csv", *[f"{method}_predictions.csv" for method in METHODS]]
missing = [name for name in required if not (OUTPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Run is incomplete; missing: {missing}")
print("Prediction files ready:", len(required), "/", len(required))


API key loaded from .env (hidden). Fresh run: 20260819T005742_616204Z
Shared dynamic-agent outputs: 16/357
Shared dynamic-agent outputs: 32/357
Shared dynamic-agent outputs: 48/357
Shared dynamic-agent outputs: 64/357
Shared dynamic-agent outputs: 80/357
Shared dynamic-agent outputs: 96/357
Shared dynamic-agent outputs: 112/357
Shared dynamic-agent outputs: 128/357
Shared dynamic-agent outputs: 144/357
Shared dynamic-agent outputs: 160/357
Shared dynamic-agent outputs: 176/357
Shared dynamic-agent outputs: 192/357
Shared dynamic-agent outputs: 208/357
Shared dynamic-agent outputs: 224/357
Shared dynamic-agent outputs: 240/357
Shared dynamic-agent outputs: 256/357
Shared dynamic-agent outputs: 272/357
Shared dynamic-agent outputs: 288/357
Shared dynamic-agent outputs: 304/357
Shared dynamic-agent outputs: 320/357
Shared dynamic-agent outputs: 336/357
Shared dynamic-agent outputs: 352/357
Shared dynamic-agent outputs: 357/357
Evaluating: jsd
Evaluating: kendall_w
Evaluating: krippendorff

In [5]:
# Cell 2 - Validate final outcomes, models, and compute all method summaries
def summarize(filename, system, agreement_method):
    frame = pd.read_csv(OUTPUT_DIR / filename, keep_default_na=False)
    test = frame[frame.split == "test"].copy()
    assert len(test) == 300 and test.case_id.astype(str).is_unique
    assert set(test.safety_model.astype(str)) == {FINAL_SAFETY_MODEL}
    labels = sorted(set(test.gold_answer.astype(str)) | set(test.predicted_answer.astype(str)))
    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        test.gold_answer, test.predicted_answer, labels=labels, average="macro", zero_division=0
    )
    _, _, weighted_f1, _ = precision_recall_fscore_support(
        test.gold_answer, test.predicted_answer, labels=labels, average="weighted", zero_division=0
    )
    return {
        "System": system, "Agreement Method": agreement_method, "n": 300,
        "Accuracy": accuracy_score(test.gold_answer, test.predicted_answer),
        "Macro Precision": precision, "Macro Recall": recall, "Macro F1": macro_f1,
        "Weighted F1": weighted_f1,
        "GPT-5 Safety Violation Rate": test.safety_violation.astype(bool).mean(),
        "Disagreement Trigger Rate": test.get("disagreement_triggered", pd.Series(False, index=test.index)).astype(bool).mean(),
        "Judge Trigger Rate": test.get("judge_triggered", pd.Series(False, index=test.index)).astype(bool).mean(),
        "Judge Changed Answer Rate": test.get("judge_changed_answer", pd.Series(False, index=test.index)).astype(bool).mean(),
        "Final Changed From Majority Rate": test.get("final_changed_from_majority", pd.Series(False, index=test.index)).astype(bool).mean(),
    }


labels = {
    "jsd": "JSD", "kendall_w": "Kendall W",
    "krippendorff_alpha": "Krippendorff Alpha",
    "care_consensus": "CARE v2 Calibrated Clinical Guard",
}
summaries = [summarize("single_predictions.csv", "Single GPT-4.1", "None")]
summaries += [
    summarize(f"{method}_predictions.csv", "Dynamic Multi-Agent", labels[method])
    for method in METHODS
]

# Compact routing audit: proves stronger models were dynamically allocated.
records = [json.loads(line) for line in (OUTPUT_DIR / "shared_agent_outputs.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
routing_audit = pd.DataFrame([{
    "difficulty": record["route"]["difficulty"],
    "importance": record["route"]["clinical_importance"],
    "lead_specialty": record["route"]["lead_specialty"],
    "reasoning_specialists": record["model_plan"]["specialist_models"].count(COMPLEX_REASONING_MODEL),
    "judge_model": record["model_plan"]["judge_model"],
} for record in records])
display(routing_audit.groupby(["difficulty", "importance", "reasoning_specialists", "judge_model"]).size().rename("cases").reset_index())

# A tie in accuracy is valid when different trigger sets lead to the same final answers.
method_frames = {
    method: pd.read_csv(OUTPUT_DIR / f"{method}_predictions.csv", keep_default_na=False)
    for method in METHODS
}
prediction_differences = []
for left, right in combinations(METHODS, 2):
    a = method_frames[left].query("split == 'test'").sort_values("case_id")
    b = method_frames[right].query("split == 'test'").sort_values("case_id")
    prediction_differences.append({
        "Method A": labels[left], "Method B": labels[right],
        "Different Final Answers": int((a.predicted_answer.to_numpy() != b.predicted_answer.to_numpy()).sum()),
        "Different Judge Triggers": int((a.judge_triggered.to_numpy() != b.judge_triggered.to_numpy()).sum()),
    })
display(pd.DataFrame(prediction_differences))


,difficulty,importance,reasoning_specialists,judge_model,cases
0,complex,critical,2,gpt-5-mini,2
1,complex,high,1,gpt-5-mini,1
2,moderate,critical,2,gpt-5-mini,24
3,moderate,high,1,gpt-5-mini,120
4,moderate,routine,0,gpt-4.1,2
5,simple,critical,2,gpt-5-mini,10
6,simple,high,1,gpt-5-mini,107
7,simple,routine,0,gpt-4.1,91


,Method A,Method B,Different Final Answers,Different Judge Triggers
0,JSD,Kendall W,6,72
1,JSD,Krippendorff Alpha,10,12
2,JSD,CARE v2 Calibrated Clinical Guard,7,30
3,Kendall W,Krippendorff Alpha,8,68
4,Kendall W,CARE v2 Calibrated Clinical Guard,7,82
5,Krippendorff Alpha,CARE v2 Calibrated Clinical Guard,7,18


In [6]:
# Final Cell - Main 300-test accuracy, F1, and GPT-5 safety comparison
comparison = pd.DataFrame(summaries).sort_values(
    ["Accuracy", "Weighted F1", "Macro F1", "GPT-5 Safety Violation Rate"],
    ascending=[False, False, False, True],
).reset_index(drop=True)
rank_key = comparison[["Accuracy", "Weighted F1", "Macro F1"]].round(12).apply(tuple, axis=1)
comparison.insert(0, "Rank", pd.factorize(rank_key)[0] + 1)
comparison.to_csv(OUTPUT_DIR / "final_main_comparison.csv", index=False)
(ROOT / "agreement-experiment" / "LATEST_FRESH_RUN.txt").write_text(
    str(OUTPUT_DIR), encoding="utf-8"
)
if ACTIVE_RUN_FILE.exists() and ACTIVE_RUN_FILE.read_text(encoding="utf-8").strip() == str(OUTPUT_DIR):
    ACTIVE_RUN_FILE.unlink()
display(comparison.style.format({
    "Accuracy": "{:.4f}", "Macro Precision": "{:.4f}", "Macro Recall": "{:.4f}",
    "Macro F1": "{:.4f}", "Weighted F1": "{:.4f}",
    "GPT-5 Safety Violation Rate": "{:.4f}",
    "Disagreement Trigger Rate": "{:.4f}", "Judge Trigger Rate": "{:.4f}",
    "Judge Changed Answer Rate": "{:.4f}",
    "Final Changed From Majority Rate": "{:.4f}",
}))


,Rank,System,Agreement Method,n,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1,GPT-5 Safety Violation Rate,Disagreement Trigger Rate,Judge Trigger Rate,Judge Changed Answer Rate,Final Changed From Majority Rate
0,1,Dynamic Multi-Agent,CARE v2 Calibrated Clinical Guard,300,0.7667,0.8499,0.8423,0.8439,0.7672,0.0200,0.2367,0.2367,0.0467,0.0533
1,2,Dynamic Multi-Agent,JSD,300,0.7633,0.8470,0.8405,0.8420,0.7642,0.0200,0.1367,0.1367,0.0400,0.0400
2,3,Dynamic Multi-Agent,Kendall W,300,0.7600,0.8461,0.8385,0.8403,0.7608,0.0200,0.3400,0.3433,0.0467,0.0467
3,4,Dynamic Multi-Agent,Krippendorff Alpha,300,0.7500,0.8406,0.8311,0.8325,0.7500,0.0233,0.1767,0.1767,0.0467,0.0467
4,5,Single GPT-4.1,None,300,0.7200,0.6509,0.6469,0.6460,0.7187,0.0300,0.0000,0.0000,0.0000,0.0000
